# 02. 搜索策略与 ReWOO

**ReWOO** = **Re**asoning **W**ith**O**ut **O**bservation。与 ReAct 每步都等待工具结果不同，ReWOO 先生成完整计划（含占位符），再批量执行工具填充结果。

**优势**：减少 LLM 调用次数（工具结果不打断推理链），适合工具依赖关系可预测的场景。

**代价**：如果工具结果改变了计划的前提条件，ReWOO 无法及时调整。

> **检查点**：能说出 ReWOO 比 ReAct 更适合的一个具体场景，以及 ReAct 比 ReWOO 更优的一个具体场景。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# ReWOO 风格：占位符计划 + 批量执行
from dataclasses import dataclass

@dataclass
class PlanStep:
    id: str; tool: str; args: dict; output_var: str

def create_plan(goal: str) -> list[PlanStep]:
    if "weather" in goal.lower():
        return [
            PlanStep("S1","weather",{"city":"Beijing"},"weather_beijing"),
            PlanStep("S2","weather",{"city":"Shanghai"},"weather_shanghai"),
            PlanStep("S3","compare",{"a":"$weather_beijing","b":"$weather_shanghai"},"result"),
        ]
    return [PlanStep("S1","search",{"query":goal},"result")]

def execute_plan(steps: list[PlanStep]) -> dict:
    ctx = {}
    tools = {
        "weather": lambda **kw: f"{kw['city']}: 25°C",
        "compare": lambda **kw: f"{kw['a']} vs {kw['b']}: same temp",
        "search": lambda **kw: f"Results for {kw['query']}: found 3 docs",
    }
    for s in steps:
        args = {k: (ctx.get(v[1:],v) if isinstance(v,str) and v.startswith("$") else v)
                for k,v in s.args.items()}
        result = tools[s.tool](**args)
        ctx[s.output_var] = result
        print(f"{s.id}: {s.tool}({args}) -> {s.output_var} = {result[:50]}")
    return ctx

plan = create_plan("compare weather")
print(f"Plan: {len(plan)} steps")
ctx = execute_plan(plan)
print(f"\nFinal: {ctx.get('result','unknown')}")

Plan: 3 steps
S1: weather({'city': 'Beijing'}) -> weather_beijing = Beijing: 25°C
S2: weather({'city': 'Shanghai'}) -> weather_shanghai = Shanghai: 25°C
S3: compare({'a': 'Beijing: 25°C', 'b': 'Shanghai: 25°C'}) -> result = Beijing: 25°C vs Shanghai: 25°C: same temp

Final: Beijing: 25°C vs Shanghai: 25°C: same temp


## 三种范式对比

| | ReAct | Plan-Act | ReWOO |
|---|-------|----------|-------|
| 规划时机 | 逐步（每步后重规划） | 一次性（执行前） | 一次性（执行前） |
| 工具调用 | 交织在推理中 | 按序执行 | 批量执行 |
| 适应性 | 高（Observation 即时反馈） | 中（可 re-plan） | 低（计划固定） |
| Token 效率 | 低 | 中 | 高 |

> **检查点**：针对一个具体任务，能论证你选择的范式及其权衡。

## 真实 API 实验

对比「穷举生成 + 启发式剪枝」与「LLM 生成候选」在 24 点上的搜索效率：谁用更少节点到达解？


In [3]:
# LLM 候选生成的质量观察：给同一状态，LLM 提议 3 步 vs 穷举 24+ 步
data = chat_json(client, """24点游戏：当前数字 [4, 7, 8, 8]，目标 24。
提议 3 个不同的下一步（两数运算）。返回 JSON: {"moves": [{"a": 4, "b": 7, "op": "+"}, ...]}""")
for mv in data.get("moves", []):
    print(f"  LLM 提议: {mv['a']} {mv['op']} {mv['b']}")
print("\n对比：穷举 generate_children 会产生 ~24 个候选，LLM 只产生 3 个但含「直觉」。")
print("这就是 ToT 的核心交换：用 LLM 的领域直觉替代暴力枚举，省节点但可能漏解。")


  LLM 提议: 4 + 7
  LLM 提议: 8 - 4
  LLM 提议: 8 * 8

对比：穷举 generate_children 会产生 ~24 个候选，LLM 只产生 3 个但含「直觉」。
这就是 ToT 的核心交换：用 LLM 的领域直觉替代暴力枚举，省节点但可能漏解。


## 练习

1. 扩展 ReWOO 的 plan：如果 `compare` 结果不令人满意（温度差异大），增加一个额外的 `suggest` 步骤推荐出行建议。
2. 实现一个混合模式：先用 ReWOO 规划，但如果某个工具执行结果与预期不符（添加 `validate` 检查），回退到 ReAct 模式重新探索。
3. 思考 ReWOO 在什么情况下会产生比 ReAct **更差**的结果？提示：考虑信息依赖关系。

> **检查点**：能在 Plan-Act、ReAct、ReWOO 之间做出基于场景的理性选择，而非盲目选用一种。